<a href="https://colab.research.google.com/github/thienandinh29/nghiemcuukhoahocsgu2026/blob/main/data_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# 1. Load data (Giữ nguyên phần load dữ liệu của bạn)
df_full = pd.read_csv('sp500_supervised_features_final.csv')
if 'Date' in df_full.columns:
    df_full['Date'] = pd.to_datetime(df_full['Date'])
    df_full.set_index('Date', inplace=True)
df_full = df_full.sort_index()

target_cols = ['Target_1D', 'Target_5D']
feature_cols = [col for col in df_full.columns if col not in target_cols]

# 2. Khởi tạo tham số cho Cửa sổ trượt (Walk-Forward Validation)
# Giả sử: Học từ 3 năm quá khứ, Dự báo cho 1 năm tiếp theo (Có thể tùy chỉnh)
train_years = 3
test_years = 1
start_year = df_full.index.year.min() # 2010
end_year = df_full.index.year.max()   # 2025

# Khởi tạo Dataframe để lưu toàn bộ kết quả dự báo ngoài mẫu (Out-of-sample)
oos_predictions = pd.DataFrame()

print("--- BẮT ĐẦU HUẤN LUYỆN CỬA SỔ TRƯỢT (ROLLING WINDOW) ---")

# 3. Vòng lặp Cửa sổ trượt
for current_year in range(start_year, end_year - train_years + 1, test_years):

    # Xác định mốc thời gian cho cửa sổ hiện tại
    train_start = f"{current_year}-01-01"
    train_end = f"{current_year + train_years - 1}-12-31"

    test_start = f"{current_year + train_years}-01-01"
    test_end = f"{current_year + train_years + test_years - 1}-12-31"

    # Cắt dữ liệu
    train_df = df_full.loc[train_start:train_end].copy()
    test_df = df_full.loc[test_start:test_end].copy()

    # Nếu tập test trống (vượt quá năm hiện tại) thì dừng lặp
    if test_df.empty:
        break

    print(f"Cửa sổ: Train [{train_start} -> {train_end}] | Test [{test_start} -> {test_end}]")

    # 4. Chuẩn hóa dữ liệu (QUAN TRỌNG: Fit scaler lại từ đầu cho mỗi cửa sổ)
    scaler = StandardScaler()

    X_train = scaler.fit_transform(train_df[feature_cols])
    y_train = train_df['Target_1D']

    X_test = scaler.transform(test_df[feature_cols])
    y_test = test_df['Target_1D']

    # 5. Khởi tạo và Huấn luyện mô hình (Ví dụ với Random Forest)
    # Lưu ý: Nếu có Tuning (RandomizedSearchCV), bạn đặt nó vào bước này.
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_train, y_train)

    # 6. Dự báo Xác suất và áp dụng Quy tắc Ngưỡng 50%
    # Lấy xác suất của lớp 1 (Tăng)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    # Quy tắc ra quyết định: Xác suất > 0.5 thì Mua (1), ngược lại Đứng ngoài (0)
    y_pred_threshold = np.where(y_pred_proba > 0.5, 1, 0)

    # 7. Lưu kết quả của cửa sổ hiện tại
    window_results = pd.DataFrame(index=test_df.index)
    window_results['Actual_Target'] = y_test
    window_results['Predicted_Prob'] = y_pred_proba
    window_results['Predicted_Signal'] = y_pred_threshold

    # Gộp vào bảng kết quả tổng
    oos_predictions = pd.concat([oos_predictions, window_results])

print("\n--- HOÀN TẤT ---")
print(f"Tổng số ngày dự báo thực chiến (Out-of-sample): {len(oos_predictions)} ngày")

# Lưu dữ liệu train_df và test_df từ cửa sổ cuối cùng
print("\n--- LƯU DỮ LIỆU TRAIN VÀ TEST CUỐI CÙNG ---")
try:
    train_df.to_csv('sp500_train_final_window.csv')
    test_df.to_csv('sp500_test_final_window.csv')
    print("Dữ liệu train và test từ cửa sổ cuối cùng đã được lưu thành 'sp500_train_final_window.csv' và 'sp500_test_final_window.csv'.")
except Exception as e:
    print(f"Không thể lưu dữ liệu train/test cuối cùng: {e}")

--- BẮT ĐẦU HUẤN LUYỆN CỬA SỔ TRƯỢT (ROLLING WINDOW) ---
Cửa sổ: Train [2010-01-01 -> 2012-12-31] | Test [2013-01-01 -> 2013-12-31]
Cửa sổ: Train [2011-01-01 -> 2013-12-31] | Test [2014-01-01 -> 2014-12-31]
Cửa sổ: Train [2012-01-01 -> 2014-12-31] | Test [2015-01-01 -> 2015-12-31]
Cửa sổ: Train [2013-01-01 -> 2015-12-31] | Test [2016-01-01 -> 2016-12-31]
Cửa sổ: Train [2014-01-01 -> 2016-12-31] | Test [2017-01-01 -> 2017-12-31]
Cửa sổ: Train [2015-01-01 -> 2017-12-31] | Test [2018-01-01 -> 2018-12-31]
Cửa sổ: Train [2016-01-01 -> 2018-12-31] | Test [2019-01-01 -> 2019-12-31]
Cửa sổ: Train [2017-01-01 -> 2019-12-31] | Test [2020-01-01 -> 2020-12-31]
Cửa sổ: Train [2018-01-01 -> 2020-12-31] | Test [2021-01-01 -> 2021-12-31]
Cửa sổ: Train [2019-01-01 -> 2021-12-31] | Test [2022-01-01 -> 2022-12-31]
Cửa sổ: Train [2020-01-01 -> 2022-12-31] | Test [2023-01-01 -> 2023-12-31]
Cửa sổ: Train [2021-01-01 -> 2023-12-31] | Test [2024-01-01 -> 2024-12-31]
Cửa sổ: Train [2022-01-01 -> 2024-12-31] | 